# LLMs on Clinical notes for cannabis use patterns in ARD 
## Snippet Creation for Clinical Notes 

Based on Nathan's script snippet_creation.py
Add: create a small pipeline testing sample
add: categories based on the diagnosis


*yiyu wang 2025/02*

## 1. Create use status snippets


1 = Not true mention
2 = Denial of Use
3 = Positive Past Use
4 = Positive Current Use

In [ ]:

import pandas as pd
import re
import string
def remove_emojis(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def clean_input_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', '', text)

    # Preserve : - / #
    keep = ":/-#.,"
    remove_punct = ''.join(c for c in string.punctuation if c not in keep)
    text = text.translate(str.maketrans('', '', remove_punct))

    text = text.replace('@', '').replace('\n', '').replace('¬ø', '')

    # Tokenize with regex while preserving #### blocks
    tokens = re.findall(r'####|\w+|[:/\-]', text)
    return ' '.join(tokens)

In [ ]:

def combine_status_labels(label):
    # combine Label 1 and 2 into 1
    if label in [1, 2]:
        return 1
    elif label in [3, 4, 5]:
        return label - 1
    else:
        raise ValueError("Label must be in the range 1-5")



In [ ]:

# combine with two other active learning datasets

df = pd.read_csv('/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data/edited_Aim2_400_labels.csv')
df1 = pd.read_csv('/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data/active_learning_2.csv')
df2 = pd.read_csv('/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data/active_learning_3.csv')
# combined with the rest 
# process such that Label is the first digit from the sentiment column
df1['Label'] = df1['sentiment'].str.extract('(\d)').astype(int)
df1['Snippets'] = df1['text'].apply(clean_input_text)

df2['Label'] = df2['sentiment'].str.extract('(\d)').astype(int)
df2['Snippets'] = df2['text'].apply(clean_input_text)

df1['Label'] = df1['Label'].apply(combine_status_labels)
df2['Label'] = df2['Label'].apply(combine_status_labels)


# process such that Label is the first digit from the sentiment column
df['Label'] = df['sentiment'].str.extract('(\d)').astype(int)
# remove non word punctuation from the 'Snippet' column
df['Snippets'] = df['Snippets'].apply(clean_input_text)
df['Label'] = df['Label'].apply(combine_status_labels)


# combine the with df, and use the combined_df with the sampled_df to make sure there is no leaking between train and validation
combined_df = pd.concat([df, df1, df2], ignore_index=True).reset_index(drop=True)
combined_df['Label'].value_counts().sort_index()

# calcaulate the percentage of each label
combined_df['Label'].value_counts(normalize=True).sort_index() * 100

combined_df['index'] = combined_df.index + 1

# check duplicates based on the 'Snippets' column
duplicates = combined_df.duplicated(subset=['Snippets'], keep=False)
print(f"Number of snippets with duplicates: {duplicates.sum()}")
# remove duplicates based on the 'Snippets' column, keeping the first occurrence
combined_df = combined_df.drop_duplicates(subset=['Snippets'], keep='first')

# save this as the master csv
combined_df.to_csv('/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data/use_case_master.csv', index=False)

print(combined_df.shape)

In [ ]:
#get the distribution of the classes

import numpy as np
labels, counts = np.unique(combined_df['Label'], return_counts=True)
# print the percentage for the notes
for (label, count) in zip(labels, counts):
    print(f"label {label}: n = {count}, {count/len(combined_df)*100}%")
    


In [ ]:
# remove the snippets used for prompting shots
prompting_snippets =[
    "available which may be helpful some report finding #### cbd #### topical to be helpful hand therapy maybe helpful if",
    "2 y and did not use it beyond that incident pt denied #### marijuana #### and illicit substance use pt denied misuse / dependen",
    "rgies social history : married former smoker history of #### mj #### family history : no ra or sle no psoriasis no ibd no cv",
    "hair loss oral ulcers anxiety related sob - now on #### cbd #### prior rheum history : for her chest pain this resulte",
    "call the pain clinic and talk to them about getting the #### cbd #### movements classes she will do at vi pt will accept",
    "2 - 3 pack - year history he is looking into medical #### marijuana #### but is not currently using he has never done illicit",
    "smoke any tobacco he does not do any drugs including #### marijuana #### but has experimented with this he works at google in",
    "ng percocet and recently higher dose oxycontin taking #### cbd #### tincture 5 mg every three hrs and this helps she is h"]



# remove these snippets from combined_df
combined_df = combined_df[~combined_df['Snippets'].isin(prompting_snippets)]

# Include the additional annotations when estimating prevalence, while keeping
# those rows reserved for training as in the original workflow.
additional_annotated_data = pd.read_csv(
    '/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data/manual_annotation_sampled_100_snippets_YW_combined.csv'
)
additional_annotated_data.rename(
    columns={'label_YW': 'Label', 'Relevant Snippets': 'Snippets'},
    inplace=True,
)
annotation_population = pd.concat(
    [combined_df, additional_annotated_data], ignore_index=True
)
annotation_population_prevalence = (
    annotation_population['Label'].value_counts(normalize=True).sort_index()
)
print(combined_df.shape)
print(annotation_population_prevalence)

In [ ]:
df_sampled = pd.read_csv('/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data/set-validation_task-status_label-4_sample-10_desc-stratified.csv')


# df_sampled = combined_df.groupby('Label').apply(lambda x: x.sample(n=10, random_state=42, replace=False)).reset_index(drop=True)
# df_sampled.to_csv('/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data/set-validation_task-status_label-4_sample-10_desc-stratified.csv', index=False)
print(df_sampled.shape)

In [ ]:
# remove the snippets 
for snippet in df_sampled['Snippets']:
    combined_df = combined_df[combined_df['Snippets'] != snippet]

print(combined_df.shape)

In [ ]:
# Create two alternative test sets with the same total size:
# 1) balanced: the same number of rows from every class
# 2) representative: class counts follow the full annotation population
random_state = 42
n_per_class = 10
data_dir = '/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data'

df_test_balanced = (
    combined_df.groupby('Label', group_keys=False)
    .apply(lambda group: group.sample(n=n_per_class, random_state=random_state, replace=False))
    .reset_index(drop=True)
)
test_size = len(df_test_balanced)

# Convert population prevalences to integer counts with the largest-remainder method.
expected_counts = annotation_population_prevalence * test_size
representative_counts = expected_counts.astype(int)
rows_left = test_size - representative_counts.sum()
if rows_left:
    fractional_remainders = (expected_counts - representative_counts).sort_values(ascending=False)
    representative_counts.loc[fractional_remainders.index[:rows_left]] += 1

available_counts = combined_df['Label'].value_counts()
available_for_each_class = available_counts.reindex(representative_counts.index, fill_value=0)
if (representative_counts > available_for_each_class).any():
    raise ValueError('Not enough rows in one or more classes for the representative test set.')

df_test_representative = pd.concat(
    [
        combined_df[combined_df['Label'] == label].sample(
            n=int(count), random_state=random_state, replace=False
        )
        for label, count in representative_counts.items()
    ],
    ignore_index=True,
)

df_test_balanced.to_csv(
    f'{data_dir}/set-test_task-status_label-4_sample-{n_per_class}-per-class_desc-balanced.csv',
    index=False,
)
df_test_representative.to_csv(
    f'{data_dir}/set-test_task-status_label-4_sample-{test_size}_desc-representative.csv',
    index=False,
)

print('Balanced test set:', df_test_balanced.shape)
print(df_test_balanced['Label'].value_counts().sort_index())
print('Representative test set:', df_test_representative.shape)
print(df_test_representative['Label'].value_counts().sort_index())

In [ ]:
# Create a leakage-free training complement for each alternative test set.
df_train_balanced = combined_df[
    ~combined_df['Snippets'].isin(df_test_balanced['Snippets'])
].copy()
df_train_representative = combined_df[
    ~combined_df['Snippets'].isin(df_test_representative['Snippets'])
].copy()

# Use the representative pair as the primary/backwards-compatible df_train and df_test.
df_test = df_test_representative.copy()
df_train = df_train_representative.copy()
combined_df = df_train
print('Balanced training set:', df_train_balanced.shape)
print('Representative training set:', df_train_representative.shape)
df_train

In [ ]:
# Check that validation and each train/test pair are leakage-free.
print("Balanced train/validation overlap:",
      len(set(df_train_balanced['Snippets']).intersection(set(df_sampled['Snippets']))))
print("Balanced train/test overlap:",
      len(set(df_train_balanced['Snippets']).intersection(set(df_test_balanced['Snippets']))))

print("Representative train/validation overlap:",
      len(set(df_train_representative['Snippets']).intersection(set(df_sampled['Snippets']))))
print("Representative train/test overlap:",
      len(set(df_train['Snippets']).intersection(set(df_test_representative['Snippets']))))



In [ ]:
# Compare the class distributions of the population and both test sets.
distribution_comparison = pd.concat(
    {
        'annotation_population_pct': annotation_population_prevalence * 100,
        'balanced_test_pct': df_test_balanced['Label'].value_counts(normalize=True).sort_index() * 100,
        'representative_test_pct': df_test_representative['Label'].value_counts(normalize=True).sort_index() * 100,
    },
    axis=1,
).fillna(0)
distribution_comparison

In [ ]:
combined_df.head()

In [ ]:
# Add the additional annotated data, which were included in the population
# prevalence calculation above and intentionally reserved for training.
reserved_balanced = set(df_sampled['Snippets']) | set(df_test_balanced['Snippets'])
reserved_representative = set(df_sampled['Snippets']) | set(df_test_representative['Snippets'])
additional_train_balanced = additional_annotated_data[
    ~additional_annotated_data['Snippets'].isin(reserved_balanced)
]
additional_train_representative = additional_annotated_data[
    ~additional_annotated_data['Snippets'].isin(reserved_representative)
]
df_train_balanced = pd.concat(
    [df_train_balanced, additional_train_balanced], ignore_index=True
).drop_duplicates(subset=['Snippets'], keep='first')
df_train_representative = pd.concat(
    [df_train_representative, additional_train_representative], ignore_index=True
).drop_duplicates(subset=['Snippets'], keep='first')
df_train = df_train_representative.copy()
combined_df = df_train
print('Balanced training set after additional annotations:', df_train_balanced.shape)
print('Representative training set after additional annotations:', df_train.shape)


In [ ]:
# check the distribution between different classes
print(combined_df['Label'].value_counts(normalize=True).sort_index() * 100)
print(combined_df['Label'].value_counts(normalize=False))

In [ ]:
df_train_balanced.to_csv(
    f'{data_dir}/set-train_task-status_label-4_sample-{len(df_train_balanced)}_desc-balanced.csv',
    index=False,
)
df_train_representative.to_csv(
    f'{data_dir}/set-train_task-status_label-4_sample-{len(df_train_representative)}_desc-representative.csv',
    index=False,
)


## 3. Create use reason snippets: label-5

In [ ]:
# create use case csv
# combine three datasets into one
df = pd.DataFrame()

for f in ['/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data/use_case_active_learning_1.csv',
        '/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data/use_case_active_learning_2.csv',
        '/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data/use_case_active_learning_3.csv']:
    df_temp = pd.read_csv(f)
    df = pd.concat([df, df_temp], ignore_index=True)
    # df = df.drop_duplicates(subset=['Snippets'])
   
# combine 6 and 7 in Sentiment
# drop redundant Snippets rows

df.reset_index(drop=True, inplace=True)
df['Sentiment'] = df['Sentiment'].replace({5: 6})
df['Sentiment'] = df['Sentiment'].replace({7: 5})
print(df['Sentiment'].value_counts())
df['Label']= df['Sentiment']
df['Snippets'] = df['Snippets'].apply(clean_input_text)
df['index'] = df.index + 1

# check duplicates based on the 'Snippets' column
duplicates = df.duplicated(subset=['Snippets'], keep=False)
print(f"Number of snippets with duplicates: {duplicates.sum()}")
# remove duplicates based on the 'Snippets' column, keeping the first occurrence
df = df.drop_duplicates(subset=['Snippets'], keep='first')

print(df.shape)


In [ ]:

import numpy as np
labels, counts = np.unique(df['Label'], return_counts=True)
# print the percentage for the notes
for (label, count) in zip(labels, counts):
    print(f"label {label}: n = {count}, {count/len(df)*100}%")
    

In [ ]:
# remove label 5 and make label 6 as label 5
df = df[df['Label'] != 5]
df['Label'] = df['Label'].replace({6: 5})
print(df.shape)

In [ ]:

labels, counts = np.unique(df['Label'], return_counts=True)
# print the percentage for the notes
for (label, count) in zip(labels, counts):
    print(f"label {label}: n = {count}, {count/len(df)*100}%")

In [ ]:
reason_prompts = ["nate toprol xl 25 mg extended release tablet take 1 / 2 tablet 12 5mg by mouth daily other drug #### cbd #### oil at bedtime for pain current facility - administered medications : acetaminophen tyle"
    ,
    "on medication no gi distress with over - the - counter anti - inflammatories has tried to use medical #### #### marijuana #### #### for symptomatic relief but wants to stop the same in particular no history of p substance and sexual activity alcohol use : no drug use : yes frequency : 7 0 times per week types : #### #### marijuana #### #### sexual activity : yes partners : female no current outpat"
    ,
    "no hospital for nausea and vomiting no abdominal pain patient admits that she has been taking #### #### marijuana #### #### for 3 months now stating that it is the only way she can eat properly she stopped taking #### #### marijuana #### #### and started to have nausea / vomiting again patient is very tearful"
    ,
    "at he has never smoked he has never used smokeless tobacco he reports that he uses illicit drugs #### #### marijuana #### #### he reports that he does not drink alcohol meds : outpatient prescriptions mar te 2 5 mg tablet take 6 tablets by mouth every friday dont take on days of lumbar puncture 60 tab 3 other drug #### #### cannabis #### #### oil 1 drop po tid for nausea pimecrolimus elidel 1 crea b"
    ,
    "is working as a caregiver for a 93 - year - old woman in saratoga no alcohol or tobacco she smokes #### #### marijuana #### #### at night for anxiety and insomnia exam : current vital signs temp : 36 7 c 98"
    ,
    "ivity alcohol use : not currently drug use : yes frequency : 7 0 times per week types : #### #### marijuana #### #### comment : use 5 ml to go to sleep - edible gummy sexual activity : not curren"
    ,
    "s empathy provided spoke with patient regarding idt recommendations patient confirmed she uses #### #### cannabis #### #### most evening to calm herself down recommended she consider a referral to pain mana"
    ,
    "ng on diet eats wheat minimizes it 1 cup of coffee per day one glass of wine per day uses #### #### marijuana #### #### daily this helps her stomach 5 miscarriages active medical problems : e per week comment : 3 - 4 standard drinks per week drug use : yes frequency : 2 0 times per week types : #### #### marijuana #### #### comment : smoking #### mj #### 4 - 5 times a week helps relax sexual activity : yes"
    ,
    "and sexual activity alcohol use : not currently comment : occasional drug use : yes types : #### #### marijuana #### #### comment : occasional sexual activity : never partners : male birth control"
    ,
    "stinging misc by misc non - drug combo route route 3 times a day tablet 3 daily other drug #### #### cannabis #### #### tincture 1 : 1 #### cbd #### / #### thc #### and topical ranitidine zantac 75 75 mg tabs take by mouth d his is not disease modifying - labs today then again in a month then follow - up 3 months - she will also continue her medical #### cbd #### 30 min visit with 50 time in face - to - face discussio"]


In [ ]:
# Remove prompting examples and retain the prevalence of the full annotated
# reason population before the validation rows are held out.
print(len(reason_prompts))
df = df[~df['Snippets'].isin(reason_prompts)]
reason_annotation_population_prevalence = (
    df['Label'].value_counts(normalize=True).sort_index()
)
print(df.shape)
print(reason_annotation_population_prevalence)

In [ ]:
# sample snippets with equal numbers across all cases (Sentiment)
# df_10 = df.groupby('Sentiment').apply(lambda x: x.sample(n=10, random_state=42, replace=False)).reset_index(drop=True)
# df_10.to_csv('/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data/set-validation_task-reason_label-6_sample-10_desc-stratified.csv', index=False)
# df_10

df_10 = pd.read_csv('/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data/set-validation_task-reason_label-6_sample-10_desc-stratified.csv')
# remove label 5 and make label 6 as label 5
df_10 = df_10[df_10['Label'] != 5]
df_10['Label'] = df_10['Label'].replace({6: 5})
print(df_10.shape)
# save
df_10.to_csv('/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data/set-validation_task-reason_label-5_sample-10_desc-stratified.csv', index=False)


In [ ]:
# remove the sampled snippets from the training set
df = df[~df['Snippets'].isin(df_10['Snippets'])]
print(df.shape)

In [ ]:

labels, counts = np.unique(df['Label'], return_counts=True)
# print the percentage for the notes
for (label, count) in zip(labels, counts):
    print(f"label {label}: n = {count}, {count/len(df)*100}%")

In [ ]:
# Create balanced and annotation-population-representative reason test sets
# with the same total number of rows.
reason_random_state = 42
reason_n_per_class = 10
reason_data_dir = '/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD/data'

df_test_reason_balanced = pd.concat(
    [
        group.sample(n=reason_n_per_class, random_state=reason_random_state, replace=False)
        for _, group in df.groupby('Label')
    ],
    ignore_index=True,
)
reason_test_size = len(df_test_reason_balanced)

# Convert prevalence targets to integer counts using the largest-remainder method.
reason_expected_counts = reason_annotation_population_prevalence * reason_test_size
reason_representative_counts = reason_expected_counts.astype(int)
reason_rows_left = reason_test_size - reason_representative_counts.sum()
if reason_rows_left:
    reason_fractional_remainders = (
        reason_expected_counts - reason_representative_counts
    ).sort_values(ascending=False)
    reason_representative_counts.loc[
        reason_fractional_remainders.index[:reason_rows_left]
    ] += 1

reason_available_counts = df['Label'].value_counts().reindex(
    reason_representative_counts.index, fill_value=0
)
if (reason_representative_counts > reason_available_counts).any():
    raise ValueError('Not enough rows in one or more reason classes for the representative test set.')

df_test_reason_representative = pd.concat(
    [
        df[df['Label'] == label].sample(
            n=int(count), random_state=reason_random_state, replace=False
        )
        for label, count in reason_representative_counts.items()
    ],
    ignore_index=True,
)

df_test_reason_balanced.to_csv(
    f'{reason_data_dir}/set-test_task-reason_label-5_sample-{reason_n_per_class}-per-class_desc-balanced.csv',
    index=False,
)
df_test_reason_representative.to_csv(
    f'{reason_data_dir}/set-test_task-reason_label-5_sample-{reason_test_size}_desc-representative.csv',
    index=False,
)

print('Balanced reason test set:', df_test_reason_balanced.shape)
print(df_test_reason_balanced['Label'].value_counts().sort_index())
print('Representative reason test set:', df_test_reason_representative.shape)
print(df_test_reason_representative['Label'].value_counts().sort_index())

In [ ]:
# Create a leakage-free training complement for each reason test set.
df_train_reason_balanced = df[
    ~df['Snippets'].isin(df_test_reason_balanced['Snippets'])
].copy()
df_train_reason_representative = df[
    ~df['Snippets'].isin(df_test_reason_representative['Snippets'])
].copy()

# Use the representative pair as the primary reason df_train and df_test.
df_test = df_test_reason_representative.copy()
df_train = df_train_reason_representative.copy()
df = df_train
print('Balanced reason training set:', df_train_reason_balanced.shape)
print('Representative reason training set:', df_train_reason_representative.shape)

In [ ]:
# Compare the annotation population with both reason test distributions.
reason_distribution_comparison = pd.concat(
    {
        'annotation_population_pct': reason_annotation_population_prevalence * 100,
        'balanced_test_pct': df_test_reason_balanced['Label'].value_counts(normalize=True).sort_index() * 100,
        'representative_test_pct': df_test_reason_representative['Label'].value_counts(normalize=True).sort_index() * 100,
    },
    axis=1,
).fillna(0)
reason_distribution_comparison

In [ ]:
print('Balanced reason train counts:')
print(df_train_reason_balanced['Label'].value_counts().sort_index())
print('Representative reason train counts:')
print(df_train_reason_representative['Label'].value_counts().sort_index())

In [ ]:
print('Reason validation set:', df_10.shape)
print('Balanced reason split:', df_train_reason_balanced.shape, df_test_reason_balanced.shape)
print(
    'Representative reason split:',
    df_train_reason_representative.shape,
    df_test_reason_representative.shape,
)

In [ ]:
# Verify validation and each reason train/test pair are leakage-free.
print("Balanced reason train/validation overlap:",
      len(set(df_train_reason_balanced['Snippets']).intersection(set(df_10['Snippets']))))
print("Balanced reason train/test overlap:",
      len(set(df_train_reason_balanced['Snippets']).intersection(set(df_test_reason_balanced['Snippets']))))
print("Representative reason train/validation overlap:",
      len(set(df_train_reason_representative['Snippets']).intersection(set(df_10['Snippets']))))
print("Representative reason train/test overlap:",
      len(set(df_train_reason_representative['Snippets']).intersection(set(df_test_reason_representative['Snippets']))))

In [ ]:
df_train_reason_balanced.to_csv(
    f'{reason_data_dir}/set-train_task-reason_label-5_sample-{len(df_train_reason_balanced)}_desc-balanced.csv',
    index=False,
)
df_train_reason_representative.to_csv(
    f'{reason_data_dir}/set-train_task-reason_label-5_sample-{len(df_train_reason_representative)}_desc-representative.csv',
    index=False,
)

In [ ]:
# quick count how many words in each snippet
df['word_count'] = df['Snippets'].apply(lambda x: len(x.split()))
df[['Snippets', 'word_count']]

# Combine all the snippets 

In [ ]:
import pandas as pd
from rapidfuzz import process, fuzz
import re
project_dir = '/Users/yiyuwang/Desktop/SNAPL/Projects/LLM_ARD'
conditions = ['SSc', 'SLE', 'SjS', 'RA', 'PsA', 'AS']
context_length = 100

In [ ]:
run_snippet_creation = True
output_path = f'{project_dir}/data/matched_rows_with_context_buffer-{context_length}_uid.csv'
if run_snippet_creation:
    files = [
    f'{project_dir}/SSc_files/clinical_note.csv',
    # f'{project_dir}/SLE_files/lupus_clinic_notes 9.12.2024.csv',
    f'{project_dir}/SLE_new/clinical_note.csv',
    f'{project_dir}/Sjs_files/SjS_files/clinical_note.csv',
    f'{project_dir}/RA_files/clinical_note.csv',
    f'{project_dir}/PsA_files/clinical_note.csv',
    f'{project_dir}/AS_files/starr_phi_confidential_stride_53750_extract_322/clinical_note.csv',

    ]

    columns_mapping = {
        'patient_id': 'Patient Id',
        'date': 'Date',
        'text': 'Scrubbed Text'
    }
    expected_columns = len(columns_mapping)
    dataframes = []

    for i, file in enumerate(files):
        df = pd.read_csv(file)
        df.rename(columns=columns_mapping, inplace=True)
        df = df[['Patient Id', 'Date', 'Scrubbed Text']]
        # Check for rows with unexpected number of columns
        df = df[df.apply(lambda x: len(x) == expected_columns, axis=1)]

        df['Condition'] = conditions[i]
        dataframes.append(df)

    data = pd.concat(dataframes, ignore_index=True).drop_duplicates()
    data = data[(data['Date'] >= '2015-01-01') & (data['Date'] <= '2024-12-31')]
    print(data["Patient Id"].nunique())
    # save the list of unique patient ids for extracting demographics later
    unique_patient_ids = data["Patient Id"].unique()
    pd.DataFrame(unique_patient_ids, columns=["Patient Id"]).to_csv(f'{project_dir}/data/unique_patient_ids.csv', index=False)

    
    print(f"Number of rows in the combined DataFrame: {data.shape[0]}")
    print(f"Number of total snippets: {data.shape[0]}")

    data['Scrubbed Text'] = data['Scrubbed Text'].astype(str)

    fuzzy_keywords = [
        "Cannabis", "Canabis," "Cannibis," "Cannibas", "Marijuana",
        "Tetrahydrocannabinol", "Cannabidiol", "Dronabinol", "Marinol", "Syndros", "Epidiolex", "Cannabigerol",
        "Nabilone", "Nabiximols", "Syndros", "Reduvo", "Cesamet", "Sativex", "Sinsemilla", "Indica", "Sativa",
    ]

    other_fuzzy = [
        "Brownies", "Tincture",
    ]

    exact_keywords = [
        "THC", "CBD", "MJ", "MMJ", "Hemp", "Bong", "CBG", "Hash", "Reefer", "Ruderalis", "Spliff", "Kush", "Weed"
    ]

    other_exact = ["Smoked", "Smokes", "Smoking", "Vape", "Vapes", "Vaped", "Vaping", "Low-dose", "Microdose",
                "High-dose", "Full-spectrum", "Joint", "Smoke",     "Edible", "Edibles", "Weed", "Pot", "Herb", "Herbs",
                "Grass", "Ganja", "Dope", "Reefer", "THC", "CBD", "MJ", "MMJ", "Hemp", "Blunt", "Pipe", "Bong", "Pen", "Pod", 
                    "Gummy", "Gummies", "Inhale", "Puff", "Hit", "Drop", "Dab", "Toke", "Hash", "Kush", "Skunk", "Bud"]

    def get_fuzzy_matches(text, keywords, threshold=90, min_word_length=4):
        if not text:
            return [], []

        matched_keywords = []
        matched_typos = []
        chunks = text.lower().split('. ')  #

        for chunk in chunks:
            if len(chunk) < 3:
                continue

            words = re.findall(r'\b\w+\b', chunk)

            for word in words:
                if len(word) < min_word_length:
                    continue
                for keyword in keywords:
                    match_score = fuzz.ratio(word, keyword.lower())
                    if match_score >= threshold:
                        if match_score < 100:
                            print(f"DEBUG: Fuzzy Match: '{word}' -> '{keyword}' (Score: {match_score})")
                        matched_keywords.append(keyword)
                        matched_typos.append(word)

        combined_matches = list(set(matched_keywords + matched_typos))
        return combined_matches, matched_typos

    data['contains_exact_match'] = data['Scrubbed Text'].str.contains(
        r'\b(?:' + '|'.join(map(re.escape, fuzzy_keywords + exact_keywords)) + r')\b',
        case=False,
        na=False
    )

    data['fuzzy_matched_words'] = data['Scrubbed Text'].apply(
        lambda x: list(get_fuzzy_matches(x, fuzzy_keywords, threshold=90)[0])
    )
    data['contains_fuzzy_match'] = data['fuzzy_matched_words'].apply(lambda x: len(x) > 0)

    def extract_surrounding_text(text, matched_words, buffer=25):
        if not text or not matched_words:
            return ""

        print(f"DEBUG: Matched words: {matched_words}")

        relevant_snippets = []
        used_ranges = set()

        highlighted_text = text

        for word in matched_words:
            highlighted_text = re.sub(
                fr'\b{re.escape(word)}\b',
                f"####{word}####",
                highlighted_text,
                flags=re.IGNORECASE
            )

        for word in matched_words:
            matches = [(m.start(), m.end()) for m in re.finditer(fr'\b{re.escape(word)}\b', text, re.IGNORECASE)]

            for start, end in matches:
                snippet_start = max(0, start - buffer)
                snippet_end = min(len(text), end + buffer)

                overlapping = False
                for existing_start, existing_end in used_ranges:
                    if not (snippet_end < existing_start or snippet_start > existing_end):
                        overlapping = True
                        merged_start = min(existing_start, snippet_start)
                        merged_end = max(existing_end, snippet_end)
                        used_ranges.remove((existing_start, existing_end))
                        used_ranges.add((merged_start, merged_end))
                        break

                if not overlapping:
                    used_ranges.add((snippet_start, snippet_end))

        for snippet_start, snippet_end in sorted(used_ranges):
            snippet = highlighted_text[snippet_start:snippet_end].strip()
            relevant_snippets.append(snippet)

        return ' ... '.join(relevant_snippets).strip()


    data['Relevant Snippets'] = data.apply(
        lambda row: extract_surrounding_text(row['Scrubbed Text'], list(row['fuzzy_matched_words']) + exact_keywords, buffer = context_length)
        if row['contains_exact_match'] or row['contains_fuzzy_match'] else "", axis=1
    )

    matched_rows_with_context = data[
        (data['contains_exact_match']) | (data['contains_fuzzy_match'])
    ]

    exclusion_words = ["grass", "dilations", "gallbladder", "dvbcilates", "pollen", "allergy", "travel",
                    "traveled", "visited", 'lived', 'home', 'live', 'lives', 'visit', 'trip', "born"]

    def should_exclude_row(snippet):
        snippet_lower = snippet.lower()
        if any(trigger in snippet_lower for trigger in ["weed", "cbd", "india"]):
            if any(exclusion in snippet_lower for exclusion in exclusion_words):
                return True
        return False

    matched_rows_with_context = matched_rows_with_context[~matched_rows_with_context['Relevant Snippets'].apply(should_exclude_row)]
    print(len(matched_rows_with_context))



    matched_rows_with_context = matched_rows_with_context.drop(columns=['Scrubbed Text'])
    # if there are redundant snippets, keep the earliest one based on Date
    matched_rows_with_context = matched_rows_with_context.sort_values(by='Date').drop_duplicates(subset
        =['Relevant Snippets'], keep='first')

    total_rows = len(data)
    exact_match_count = data['contains_exact_match'].sum()
    fuzzy_match_count = data['contains_fuzzy_match'].sum()

    print(f"Total number of rows: {total_rows}")
    print(f"Number of rows with exact matches: {exact_match_count}")
    print(f"Number of rows with fuzzy matches: {fuzzy_match_count}")
    print(f"Number of matched rows: {len(matched_rows_with_context)}")
    print(f"Matched rows with context saved to: {output_path}")
    matched_rows_with_context['uid'] = matched_rows_with_context.index + 1

    matched_rows_with_context.to_csv(output_path, index=False)


In [ ]:
import pandas as pd
df = pd.read_csv('data/matched_rows_with_context_buffer-100_uid_v2.csv')
print(df.shape)

In [ ]:
import pandas as pd
context_length = 100
output_path = f'data/matched_rows_with_context_buffer-{context_length}_uid.csv'
matched_rows_with_context = pd.read_csv(output_path)
print(f"Number of patients : {matched_rows_with_context['Patient Id'].nunique()}")


# apply clean_input_text to the 'Relevant Snippets' column
matched_rows_with_context['Relevant Snippets'] = matched_rows_with_context['Relevant Snippets'].apply(clean_input_text)


# load the annotated snippets:
status_val = pd.read_csv(f'LLM_annotation_October_16/set-validation_task-status_label-4_sample-10_desc-stratified.csv')
status_train = pd.read_csv(f'LLM_annotation_October_16/set-train_task-status_label-4.csv')
reason_val = pd.read_csv(f'LLM_annotation_October_16/set-validation_task-reason_label-6_sample-10_desc-stratified.csv')
reason_train = pd.read_csv(f'LLM_annotation_October_16/set-train_task-reason_label-6_desc-all.csv')





In [ ]:
2072960 - 43689

In [ ]:
# randomly select 100 snippets from matched_rows_with_context for manual annotation
import random
import pandas as pd
context_length = 100
output_path = f'data/matched_rows_with_context_buffer-{context_length}_uid.csv'

random.seed(42)
# filter time from 2015 to 2024
matched_rows_with_context = matched_rows_with_context[(matched_rows_with_context['Date'] >= '2015-01-01') & (matched_rows_with_context['Date'] <= '2024-12-31')]
# clean the notes
matched_rows_with_context['Relevant Snippets'] = matched_rows_with_context['Relevant Snippets'].apply(clean_input_text)
# remove those already in the validation and training sets





sampled_snippets = matched_rows_with_context.sample(n=100, random_state=42)
sampled_snippets.to_csv('data/manual_annotation_sampled_100_snippets.csv', index=False)

In [ ]:
matched_rows_with_context.head()

In [ ]:
print(matched_rows_with_context.shape)

In [ ]:
# load the annotated snippets:
status_train = pd.read_csv(f'data/set-train_task-status_label-4_desc-representative.csv')
reason_train = pd.read_csv(f'data/set-train_task-reason_label-5_desc-representative.csv')
status_val = pd.read_csv(f'data/set-validation_task-status_label-4_sample-10_desc-stratified.csv')
reason_val = pd.read_csv(f'data/set-validation_task-reason_label-5_sample-10_desc-stratified.csv')
status_test = pd.read_csv(f'data/set-test_task-status_label-4_sample-40_desc-representative.csv')
reason_test = pd.read_csv(f'data/set-test_task-reason_label-5_sample-50_desc-representative.csv')

output_path = f'data/matched_rows_with_context_buffer-{context_length}_uid.csv'
matched_rows_with_context = pd.read_csv(output_path)
print(f"Number of patients : {matched_rows_with_context['Patient Id'].nunique()}")



# remove sampled from the matched_rows_with_context
sampled_snippets = pd.read_csv('data/manual_annotation_sampled_100_snippets.csv')
matched_rows_with_context = matched_rows_with_context[~matched_rows_with_context['uid'].isin(sampled_snippets['uid'])]
print(matched_rows_with_context.shape)


In [ ]:
# save 
matched_rows_with_context.to_csv('data/matched_rows_with_context_buffer-100_uid_v2.csv', index=False)

In [ ]:
# 

In [ ]:
from rapidfuzz import fuzz, process
import re
import pandas as pd

# --- Thresholds ---
auto_threshold = 90    # auto-merge if ≥90%
review_threshold = 70  # ask for manual confirmation if between 70–89%
fallback_threshold = 50  # fallback search if <70% has no match

# --- Tracking counters ---
auto_count = 0
manual_count = 0
skipped_count = 0
no_match_count = 0

for snippet in status_val['Snippets']:
    found_match = False

    # 1️⃣ Try exact match first
    matched_row = matched_rows_with_context[
        matched_rows_with_context['Relevant Snippets'].str.contains(re.escape(snippet), na=False)
    ]
    if not matched_row.empty:
        row = matched_row.iloc[0]
        status_val.loc[status_val['Snippets'] == snippet, ['Patient Id', 'Date', 'Condition', 'uid']] = [
            row['Patient Id'], row['Date'], row['Condition'], row['uid']
        ]
        auto_count += 1
        continue

    # 2️⃣ Fuzzy match: primary attempt
    candidates = matched_rows_with_context['Relevant Snippets'].dropna().tolist()
    results = process.extract(snippet, candidates, scorer=fuzz.token_sort_ratio, limit=5)

    if not results:
        print(f"\n❌ No fuzzy match found for snippet:\n>>> {snippet[:150]}...")
        no_match_count += 1
        continue

    best_match_text, best_score, best_idx = results[0]
    candidate_row = matched_rows_with_context.iloc[best_idx]

    # 3️⃣ Auto-merge for very high confidence
    if best_score >= auto_threshold:
        status_val.loc[status_val['Snippets'] == snippet, ['Patient Id', 'Date', 'Condition', 'uid']] = [
            candidate_row['Patient Id'], candidate_row['Date'], candidate_row['Condition'], candidate_row['uid']
        ]
        print(f"\n✅ Auto-merged {best_score}% match:\n>>> {snippet[:100]}...\nMatched with:\n>>> {best_match_text[:100]}...\n")
        auto_count += 1
        continue

    # 4️⃣ Manual review for moderate confidence
    elif best_score >= review_threshold:
        print(f"\n⚠️  Possible fuzzy match ({best_score}%):")
        print(f"Snippet:\n>>> {snippet[:200]}...")
        print(f"\nCandidate:\n>>> {best_match_text[:200]}...")
        print(f"Patient Id: {candidate_row['Patient Id']}, Date: {candidate_row['Date']}, Condition: {candidate_row['Condition']}")
        user_input = input("\nAccept this match? (y/n): ").strip().lower()
        if user_input in ['y', 'yes']:
            status_val.loc[status_val['Snippets'] == snippet, ['Patient Id', 'Date', 'Condition', 'uid']] = [
                candidate_row['Patient Id'], candidate_row['Date'], candidate_row['Condition'], candidate_row['uid']
            ]
            print("✅ Merged successfully.")
            manual_count += 1
        else:
            print("❎ Skipped.")
            skipped_count += 1
        continue

    # 5️⃣ Fallback fuzzy match (50–69%)
    else:
        fallback_results = [r for r in results if r[1] >= fallback_threshold]
        if fallback_results:
            print(f"\n🔍 Fallback matches (50–69%) for snippet:\n>>> {snippet[:150]}...")
            for text, score, idx in fallback_results[:3]:
                row = matched_rows_with_context.iloc[idx]
                print(f"\n  → Similarity: {score}%")
                print(f"  Candidate: {text[:200]}...")
                print(f"  Patient Id: {row['Patient Id']}, Date: {row['Date']}, Condition: {row['Condition']}")
                user_input = input("\nAccept this fallback match? (y/n): ").strip().lower()
                if user_input in ['y', 'yes']:
                    status_val.loc[status_val['Snippets'] == snippet, ['Patient Id', 'Date', 'Condition', 'uid']] = [
                        row['Patient Id'], row['Date'], row['Condition'], row['uid']
                    ]
                    print("✅ Merged (fallback).")
                    manual_count += 1
                    found_match = True
                    break
            if not found_match:
                print("❎ No fallback match accepted.")
                skipped_count += 1
        else:
            print(f"\n❌ No match ≥{fallback_threshold}% for snippet:\n>>> {snippet[:150]}...")
            no_match_count += 1

# --- Summary report ---
print("\n🎯 Fuzzy Matching Complete!")
print(f"✅ Auto-merged: {auto_count}")
print(f"🧭 Manually accepted: {manual_count}")
print(f"🚫 Skipped: {skipped_count}")
print(f"❌ No matches found: {no_match_count}")


In [ ]:
from rapidfuzz import fuzz, process
import re
import pandas as pd

# --- Thresholds ---
auto_threshold = 90    # auto-merge if ≥90%
review_threshold = 70  # ask for manual confirmation if between 70–89%
fallback_threshold = 50  # fallback search if <70% has no match

# --- Tracking counters ---
auto_count = 0
manual_count = 0
skipped_count = 0
no_match_count = 0

for snippet in status_train['Snippets']:
    found_match = False

    # 1️⃣ Try exact match first
    matched_row = matched_rows_with_context[
        matched_rows_with_context['Relevant Snippets'].str.contains(re.escape(snippet), na=False)
    ]
    if not matched_row.empty:
        row = matched_row.iloc[0]
        status_val.loc[status_val['Snippets'] == snippet, ['Patient Id', 'Date', 'Condition', 'uid']] = [
            row['Patient Id'], row['Date'], row['Condition'], row['uid']
        ]
        auto_count += 1
        continue

    # 2️⃣ Fuzzy match: primary attempt
    candidates = matched_rows_with_context['Relevant Snippets'].dropna().tolist()
    results = process.extract(snippet, candidates, scorer=fuzz.token_sort_ratio, limit=5)

    if not results:
        print(f"\n❌ No fuzzy match found for snippet:\n>>> {snippet[:150]}...")
        no_match_count += 1
        continue

    best_match_text, best_score, best_idx = results[0]
    candidate_row = matched_rows_with_context.iloc[best_idx]

    # 3️⃣ Auto-merge for very high confidence
    if best_score >= auto_threshold:
        status_val.loc[status_val['Snippets'] == snippet, ['Patient Id', 'Date', 'Condition', 'uid']] = [
            candidate_row['Patient Id'], candidate_row['Date'], candidate_row['Condition'], candidate_row['uid']
        ]
        print(f"\n✅ Auto-merged {best_score}% match:\n>>> {snippet[:100]}...\nMatched with:\n>>> {best_match_text[:100]}...\n")
        auto_count += 1
        continue

    # 4️⃣ Manual review for moderate confidence
    elif best_score >= review_threshold:
        print(f"\n⚠️  Possible fuzzy match ({best_score}%):")
        print(f"Snippet:\n>>> {snippet[:200]}...")
        print(f"\nCandidate:\n>>> {best_match_text[:200]}...")
        print(f"Patient Id: {candidate_row['Patient Id']}, Date: {candidate_row['Date']}, Condition: {candidate_row['Condition']}")
        user_input = input("\nAccept this match? (y/n): ").strip().lower()
        if user_input in ['y', 'yes']:
            status_val.loc[status_val['Snippets'] == snippet, ['Patient Id', 'Date', 'Condition', 'uid']] = [
                candidate_row['Patient Id'], candidate_row['Date'], candidate_row['Condition'], candidate_row['uid']
            ]
            print("✅ Merged successfully.")
            manual_count += 1
        else:
            print("❎ Skipped.")
            skipped_count += 1
        continue

    # 5️⃣ Fallback fuzzy match (50–69%)
    else:
        fallback_results = [r for r in results if r[1] >= fallback_threshold]
        if fallback_results:
            print(f"\n🔍 Fallback matches (50–69%) for snippet:\n>>> {snippet[:150]}...")
            for text, score, idx in fallback_results[:3]:
                row = matched_rows_with_context.iloc[idx]
                print(f"\n  → Similarity: {score}%")
                print(f"  Candidate: {text[:200]}...")
                print(f"  Patient Id: {row['Patient Id']}, Date: {row['Date']}, Condition: {row['Condition']}")
                user_input = input("\nAccept this fallback match? (y/n): ").strip().lower()
                if user_input in ['y', 'yes']:
                    status_val.loc[status_val['Snippets'] == snippet, ['Patient Id', 'Date', 'Condition', 'uid']] = [
                        row['Patient Id'], row['Date'], row['Condition'], row['uid']
                    ]
                    print("✅ Merged (fallback).")
                    manual_count += 1
                    found_match = True
                    break
            if not found_match:
                print("❎ No fallback match accepted.")
                skipped_count += 1
        else:
            print(f"\n❌ No match ≥{fallback_threshold}% for snippet:\n>>> {snippet[:150]}...")
            no_match_count += 1

# --- Summary report ---
print("\n🎯 Fuzzy Matching Complete!")
print(f"✅ Auto-merged: {auto_count}")
print(f"🧭 Manually accepted: {manual_count}")
print(f"🚫 Skipped: {skipped_count}")
print(f"❌ No matches found: {no_match_count}")


In [ ]:
print(matched_rows_with_context.shape)

In [ ]:
# if there are redundant snippets, keep the earliest one based on Date
# matched_rows_with_context = matched_rows_with_context.sort_values(by='Date').drop_duplicates(subset=['Relevant Snippets'], keep='first')
# matched_rows_with_context

In [ ]:
# check how many are redundant snippets
# df_dup = df.drop_duplicates(subset=['Relevant Snippets'], keep='first')
# df.shape[0], df_dup.shape[0]